# Preparação do Ambiente
Rode a célula abaixo antes de começar caso já tenha rodado alguma vez para garantir que seu ambiente de arquivos está completamente limpo.


In [ ]:
# CUIDADO: Isso apaga os arquivos da pasta atual do Colab para evitar conflitos com downloads antigos.
!rm -rf *


# O Básico do Google Colab

O Google Colab é um ambiente Jupyter notebook interativo, gratuito e na nuvem.

### Atalhos Importantes:
* **Shift + Enter**: Executa a célula atual e vai para a próxima.
* **Ctrl + Enter**: Executa a célula atual e continua nela.

Vamos testar abaixo com uma célula de código!


In [ ]:
print('Olá Mundo!')


In [ ]:
a = 5
b = 10


In [ ]:
print('A soma é:', a + b)


# Machine Learning na prática


## 1 Classificador Simples

Vamos treinar nosso primeiro modelo para diferenciar Gatos de Cachorros baseado em 3 características:
1. Pelo longo?
2. Perna curta?
3. Faz miau?


In [ ]:
# Características: [pelo longo, perna curta, faz miau]
gato1 = [1, 1, 1]
gato2 = [0, 1, 0]
gato3 = [1, 0, 1]

cachorro1 = [1, 1, 0]
cachorro2 = [0, 0, 0]
cachorro3 = [1, 0, 0]

treino_x = [gato1, gato2, gato3, cachorro1, cachorro2, cachorro3] # Dados
# Gabarito: gato = 1 | cachorro = 0
treino_y = [1, 1, 1, 0, 0, 0] # Classificações


In [ ]:
from sklearn.svm import LinearSVC

modelo = LinearSVC()
# O método .fit() é onde a 'mágica' do treinamento acontece
modelo.fit(treino_x, treino_y)


In [ ]:
animal_misterioso = [0, 1, 1]

# O método .predict() faz a predição em novos dados
resultado = modelo.predict([animal_misterioso])
print('Resultado da predição (1 = Gato, 0 = Cachorro):', resultado)


In [ ]:
from sklearn.metrics import accuracy_score

# Testando a acurácia do modelo em dados que ele nunca viu
teste_x = [[1, 1, 1], [1, 1, 0], [0, 1, 0]]
teste_y = [1, 0, 0] # Gabarito correto

previsoes = modelo.predict(teste_x)
acuracia = accuracy_score(teste_y, previsoes)
print(f'Acurácia do modelo: {acuracia * 100}%')


## 2 O Perceptron

Vamos implementar um neurônio digital manualmente para entender como os pesos são ajustados.


In [ ]:
wX = 1.0
wY = 1.0
limiar = 0.5
taxa_aprendizado = 0.5

def neuronio(input_x, input_y):
    soma = (wX * input_x) + (wY * input_y)
    # Função de Ativação (Step)
    return 1 if soma >= limiar else 0

def treinar(input_x, input_y, esperado, obtido):
    global wX, wY
    erro = esperado - obtido
    wX = wX + (input_x * taxa_aprendizado * erro)
    wY = wY + (input_y * taxa_aprendizado * erro)
    print(f'Pesos atualizados -> wX: {wX:.2f} | wY: {wY:.2f}')


In [ ]:
# Dados de Treino e treinamento iterativo
dados = [
    {"x": 9.4, "y": 0.2, "class": 1},
    {"x": 1.3, "y": 7.3, "class": 0},
    {"x": 8.5, "y": 1.3, "class": 1},
    {"x": 2.4, "y": 8.1, "class": 0}
]

for d in dados:
    obtido = neuronio(d['x'], d['y'])
    treinar(d['x'], d['y'], d['class'], obtido)


## 3 O Problema Real: Imagens Complexas

Agora vamos treinar uma rede neural profunda para diferenciar Gatos de Cachorros usando imagens reais!
**Dataset**: Kaggle Cats and Dogs.


In [ ]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import os


In [ ]:
# Fazendo o download do dataset na nuvem
!curl -O https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip


In [ ]:
# Descompactando os arquivos silenciosamente (-q)
!unzip -q kagglecatsanddogs_5340.zip
!ls


### Limpeza e Pré-processamento
Nem sempre os dados vêm perfeitos. Vamos apagar imagens corrompidas.


In [ ]:
num_apagados = 0
for folder_name in ('Cat', 'Dog'):
    folder_path = os.path.join('PetImages', folder_name)
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        try:
            fobj = open(fpath, 'rb')
            is_jfif = tf.compat.as_bytes('JFIF') in fobj.peek(10)
        except Exception as e:
            continue
        finally:
            fobj.close()
        if not is_jfif:
            num_apagados += 1
            os.remove(fpath)

print(f'Total de imagens corrompidas apagadas: {num_apagados}')


In [ ]:
tamanho_imagem = (180, 180)
batch_size = 32

# Carregando os dados de Treinamento (80%)
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'PetImages',
    validation_split=0.2,
    subset='training',
    seed=1337,
    image_size=tamanho_imagem,
    batch_size=batch_size
)

# Carregando os dados de Validação (20%)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'PetImages',
    validation_split=0.2,
    subset='validation',
    seed=1337,
    image_size=tamanho_imagem,
    batch_size=batch_size
)


### Visualização dos Dados


In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title('Gato' if int(labels[i]) == 0 else 'Cachorro')
        plt.axis('off')
plt.show()


In [ ]:
data_augmentation = keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
])


### Prefetch


In [ ]:
train_ds = train_ds.prefetch(buffer_size=32)
val_ds = val_ds.prefetch(buffer_size=32)


## Modelagem


Vamos modelar uma Rede Neural Convolucional!


In [ ]:
model = keras.Sequential([
    keras.Input(shape=tamanho_imagem + (3,)),
    data_augmentation,
    tf.keras.layers.Rescaling(1.0 / 255), # Normaliza os pixels (de 0-255 para 0-1)
    
    # 1º Bloco Convolucional (Detecta linhas e bordas)
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(), # Reduz o tamanho da imagem pela metade
    
    # 2º Bloco Convolucional (Detecta formas e texturas)
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    
    # 3º Bloco Convolucional (Detecta partes complexas, ex: orelhas, focinhos)
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Flatten(), # Achata a matriz 2D para um vetor 1D
    
    # Rede Neural Clássica no final para a tomada de decisão
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2), # "Desliga" 20% dos neurônios para evitar vício (Overfitting)
    tf.keras.layers.Dense(1, activation='sigmoid') # Saída: Probabilidade (0 a 1)
])
keras.utils.plot_model(model, show_shapes=True)


## Treinamento


In [ ]:
epochs = 5

# Preparando o modelo para treinar
model.compile(
    optimizer="adam", # O algoritmo inteligente que ajusta os pesos
    loss="binary_crossentropy", # A função matemática que calcula o Erro
    metrics=["accuracy"] # Métrica humana para vermos a porcentagem de acerto
)

# Iniciando o treinamento de fato
history_aula = model.fit(
    train_ds, epochs=epochs, validation_data=val_ds, verbose=1
)


## Evaluation e Acurácia


In [ ]:
import matplotlib.pyplot as plt
plt.style.use('ggplot')

#função para visualizar o historico de treinamento da rede
def plot_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    x = range(1, len(acc) + 1)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(x, acc, 'b', label='Training acc')
    plt.plot(x, val_acc, 'r', label='Validation acc')
    plt.title('Training and validation accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(x, loss, 'b', label='Training loss')
    plt.plot(x, val_loss, 'r', label='Validation loss')
    plt.title('Training and validation loss')
    plt.legend()

plot_history(history_aula)


## Executando a rede treinada


In [ ]:
import numpy as np

# Vamos pegar 1 lote de imagens do nosso dataset de VALIDAÇÃO (que o modelo nunca viu no treino)
for images, labels in val_ds.take(1):
    img = images[0]
    gabarito = int(labels[0])
    break

plt.imshow(img.numpy().astype("uint8"))
plt.title(f"Gabarito Real: {'Cachorro' if gabarito == 1 else 'Gato'}")
plt.axis('off')
plt.show()

# passe pela rede.
img_batch = np.expand_dims(img, axis=0)
predicao = model.predict(img_batch)[0][0]

resultado = "Cachorro" if predicao > 0.5 else "Gato"
print(f"O modelo previu: {resultado} (Confiança: {predicao:.4f})")


# Para Casa: Rede mais complexa


In [ ]:
def make_model(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = tf.keras.layers.Rescaling(1.0 / 255)(x)

    # Bloco Inicial
    x = tf.keras.layers.Conv2D(128, 3, strides=2, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)

    previous_block_activation = x  # Guarda a informação para o "Atalho" (Skip Connection)

    # Construindo a profundidade em escala (Deeper Scale)
    for size in [256, 512, 728]:
        x = tf.keras.layers.Activation("relu")(x)
        # SeparableConv2D: mais eficiente e rápido que o Conv2D normal
        x = tf.keras.layers.SeparableConv2D(size, 3, padding="same")(x)
        x = tf.keras.layers.BatchNormalization()(x)

        x = tf.keras.layers.Activation("relu")(x)
        x = tf.keras.layers.SeparableConv2D(size, 3, padding="same")(x)
        x = tf.keras.layers.BatchNormalization()(x)

        x = tf.keras.layers.MaxPooling2D(3, strides=2, padding="same")(x)

        # Prepara o atalho para combinar com o tamanho final do bloco
        residual = tf.keras.layers.Conv2D(size, 1, strides=2, padding="same")(
            previous_block_activation
        )
        x = tf.keras.layers.add([x, residual])  # A Mágica: Soma o atalho! (Skip Connection)
        previous_block_activation = x  # Guarda o estado atual para o próximo loop

    # Finalização Profunda
    x = tf.keras.layers.SeparableConv2D(1024, 3, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)

    # Global Average Pooling: Tira a média inteligente em vez de "achatar" tudo
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.5)(x) # Dropout de 50% para extrema resistência
    
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs)

modelo_profundo = make_model(input_shape=tamanho_imagem + (3,), num_classes=2)
keras.utils.plot_model(modelo_profundo, show_shapes=True)


In [ ]:
epochs = 10

# Callbacks são funções executadas automaticamente no final de cada época
callbacks = [
    keras.callbacks.ModelCheckpoint("save_at_{epoch}.h5"), # Salva o modelo no meio do caminho
]

modelo_profundo.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_profundo = modelo_profundo.fit(
    train_ds, epochs=epochs, callbacks=callbacks, validation_data=val_ds, verbose=1
)


In [ ]:
plot_history(history_profundo)


### Testando o Modelo Profundo


In [ ]:
# Pegando uma imagem inédita para testar a rede poderosa
for images, labels in val_ds.take(1):
    img = images[0]
    gabarito = int(labels[0])
    break

plt.imshow(img.numpy().astype("uint8"))
plt.title(f"Gabarito Real: {'Cachorro' if gabarito == 1 else 'Gato'}")
plt.axis('off')
plt.show()

# Faz a predição com o modelo profundo!
img_batch = np.expand_dims(img, axis=0)
predicao = modelo_profundo.predict(img_batch)[0][0]

resultado = "Cachorro" if predicao > 0.5 else "Gato"
print(f"O modelo profundo previu: {resultado} (Confiança: {predicao:.4f})")
